                   CHAPTER 10 =  Creating Text Embedding Models
                   

In [ ]:
# Install or upgrade the required libraries to the specified versions
!pip install -U \

    # used for creating and training embedding models
    "sentence-transformers==3.4.1" \

    # provides pretrained Transformer models like BERT
    "transformers==4.46.3" \

    # used to download and manage pretrained models
    "huggingface-hub==0.26.2" \

    # used to load datasets such as MNLI and STSB
    "datasets==3.1.0"

               Generating Contrastive Examples

In [ ]:
   # # Load and Prepare the MNLI Training Dataset

In [ ]:
# from the dataset library import load_dataset function
from datasets import load_dataset

train_dataset = load_dataset(

    # it has collection of different nlp datasets
    "glue",

    # load the mnli dataset from GLUE collection
    "mnli",

    # give the training portion of dataset
    split="train"

).select(range(50_000))  # Selecting 50k rows

# it has 4 colums (premise , hypothesis , label , idx)
# remove idx column (no need for training)
train_dataset = train_dataset.remove_columns("idx")

In [ ]:
# it shows the realtion between the 2 sentences
train_dataset[49999]

In [ ]:
    # Create the Base Embedding Model

In [ ]:
# import Sentence transformer to create meaning vector representation for sentences
from sentence_transformers import SentenceTransformer

# Use a base model (bert base)
# create a BERT-based Sentence Transformer model.
embedding_model = SentenceTransformer('bert-base-uncased')


In [ ]:
    # Configure the Softmax Loss Function

In [ ]:
# imports the losses module from Sentence Transformers
# loss function tells the model wether the prediction was correct or wrong
from sentence_transformers import losses

# Define the loss function. In softmax loss, we will also need to explicitly set the number of labels.
train_loss = losses.SoftmaxLoss(
model=embedding_model,
sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
num_labels=3
)

In [ ]:
    # Create the STSB Evaluation Dataset

In [ ]:
# Import the evaluator used to evaluate the embedding model
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the validation split of the STSB (Semantic Textual Similarity Benchmark) dataset
val_sts = load_dataset("glue", "stsb", split="validation")

# Create an evaluator for measuring the model's performance on semantic similarity
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence in each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence in each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize similarity scores from the range 0–5 to 0–1
    scores=[score / 5 for score in val_sts["label"]],

    # Use cosine similarity to compare the sentence embeddings
    main_similarity="cosine",
)

In [ ]:
    # Configure the Training Settings

In [ ]:
# Import the class used to define the training configuration
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define the training arguments for the embedding model
args = SentenceTransformerTrainingArguments(

    # Folder where the trained model and checkpoints will be saved
    output_dir="base_embedding_model",

    # Train the model for 1 complete pass through the training dataset
    num_train_epochs=1,

    # Number of training examples processed in one batch on each device (GPU/CPU)
    per_device_train_batch_size=32,

    # Number of evaluation examples processed in one batch on each device
    per_device_eval_batch_size=32,

    # Number of warmup steps before reaching the full learning rate
    warmup_steps=100,

    # Enable mixed precision (FP16) training to reduce memory usage and speed up training
    fp16=True,

    # Evaluate the model after every 100 training steps
    eval_steps=100,

    # Display the training loss after every 100 training steps
    logging_steps=100,
)

In [ ]:
     # Train the Embedding Model with Softmax Loss

In [ ]:
# Import the trainer class used to train the embedding model
from sentence_transformers.trainer import SentenceTransformerTrainer

# Create the trainer by providing all the required components
trainer = SentenceTransformerTrainer(

    # The embedding model to be trained
    model=embedding_model,

    # The training configuration (epochs, batch size, etc.)
    args=args,

    # The dataset used for training the model
    train_dataset=train_dataset,

    # The loss function used to calculate the training loss
    loss=train_loss,

    # The evaluator used to measure the model's performance during training
    evaluator=evaluator,
)

# Start the training process
trainer.train()

In [ ]:
# Evaluate our trained model
evaluator(embedding_model)

                                  MTEB

In [ ]:
# Install the MTEB library used to evaluate embedding models
!pip install -U mteb

In [ ]:
# Install MTEB version below 2.0.0 for compatibility with the current Sentence-Transformers version
!pip install "mteb<2.0.0"

In [ ]:
    # Load the Base Embedding Model

In [ ]:
# Import the SentenceTransformer class used to load or create embedding models
from sentence_transformers import SentenceTransformer

# Load the base BERT model and create a SentenceTransformer model with mean pooling
model = SentenceTransformer("bert-base-uncased")

In [ ]:
    # Evaluate the Base Embedding Model Using MTEB

In [ ]:
# Import the MTEB class used to evaluate embedding models on benchmark tasks
from mteb import MTEB

# Create an MTEB evaluation object for the Banking77 classification task
evaluation = MTEB(tasks=["Banking77Classification"])

# Run the selected evaluation task using the embedding model
results = evaluation.run(model)

In [ ]:
print(results)

In [ ]:
                # LOSS FUNCTIONS
                # USING COSINE SIMILARITY

In [ ]:
!pip install -U "datasets==3.1.0" "huggingface-hub==0.26.2"

In [ ]:
    # Prepare the Training Dataset for Cosine Similarity Loss

In [ ]:
# Import the Dataset class and the function to load datasets
from datasets import Dataset, load_dataset

# Load the first 50,000 training examples from the MNLI dataset
# Labels: 0 = Entailment, 1 = Neutral, 2 = Contradiction
train_dataset = load_dataset(
    "glue", "mnli", split="train"
).select(range(50_000))

# Remove the unnecessary index column
train_dataset = train_dataset.remove_columns("idx")

# Convert the original labels into similarity scores
# Entailment 0 = 1 Similar
# Neutral 1 = 0 Not Similar
# Contradiction 2 = 0 Not Similar
mapping = {2: 0, 1: 0, 0: 1}

# Create a new dataset with sentence pairs and similarity labels
train_dataset = Dataset.from_dict({

    # Store the premise as the first sentence
    "sentence1": train_dataset["premise"],

    # Store the hypothesis as the second sentence
    "sentence2": train_dataset["hypothesis"],

    # Convert each label to a float similarity score (0.0 or 1.0)
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})

In [ ]:
# Uninstall the existing versions of the required libraries
!pip uninstall -y sentence-transformers transformers huggingface-hub datasets

In [ ]:
    # Create the STSB Evaluator

In [ ]:
# Import the evaluator used to evaluate the embedding model
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the validation split of the STSB (Semantic Textual Similarity Benchmark) dataset
val_sts = load_dataset("glue", "stsb", split="validation")

# Create an evaluator to measure how well the embedding model captures sentence similarity
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence in each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence in each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize the similarity scores from the range 0–5 to 0–1
    scores=[score / 5 for score in val_sts["label"]],

    # Compare the sentence embeddings using cosine similarity
    main_similarity="cosine"
)

In [ ]:
    # Train the Embedding Model Using Cosine Similarity Loss

In [ ]:
# Import the SentenceTransformer model, loss functions, trainer, and training arguments
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Load the base BERT model and create a SentenceTransformer model
embedding_model = SentenceTransformer("bert-base-uncased")

# Define the Cosine Similarity Loss used to train the embedding model
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training configuration
args = SentenceTransformerTrainingArguments(

    # Folder where the trained model and checkpoints will be saved
    output_dir="cosineloss_embedding_model",

    # Train the model for one complete pass through the dataset
    num_train_epochs=1,

    # Number of training examples processed in one batch on each device
    per_device_train_batch_size=32,

    # Number of evaluation examples processed in one batch on each device
    per_device_eval_batch_size=32,

    # Number of warmup steps before reaching the full learning rate
    warmup_steps=100,

    # Enable mixed precision (FP16) training to reduce memory usage and speed up training
    fp16=True,

    # Evaluate the model after every 100 training steps
    eval_steps=100,

    # Display the training loss after every 100 training steps
    logging_steps=100,
)

# Create the trainer by providing all the required components
trainer = SentenceTransformerTrainer(

    # The embedding model to be trained
    model=embedding_model,

    # The training configuration
    args=args,

    # The dataset used for training
    train_dataset=train_dataset,

    # The loss function used during training
    loss=train_loss,

    # The evaluator used to measure the model's performance
    evaluator=evaluator
)

# Start training the embedding model
trainer.train()

In [ ]:
# Evaluate our trained model
evaluator(embedding_model)

                  #   MULTIPLE NEGATIVES RANKING LOSS

In [ ]:
    # Prepare the Training Dataset for Multiple Negatives Ranking Loss

In [ ]:
# Import the random module for shuffling, tqdm for progress bar, and Dataset utilities
import random
from tqdm import tqdm
from datasets import Dataset, load_dataset

# Load the first 50,000 training examples from the MNLI dataset
mnli = load_dataset("glue", "mnli", split="train").select(range(50_000))

# Remove the unnecessary index column
mnli = mnli.remove_columns("idx")

# Keep only the sentence pairs labeled as Entailment (label = 0)
mnli = mnli.filter(lambda x: True if x["label"] == 0 else False)

# Create a dictionary to store the training triplets
# Anchor   -> Premise sentence
# Positive -> Correct hypothesis
# Negative -> Incorrect (random) hypothesis
train_dataset = {
    "anchor": [],
    "positive": [],
    "negative": []
}

# Copy all hypothesis sentences to create soft negatives
soft_negatives = mnli["hypothesis"]

# Randomly shuffle the hypothesis sentences
# This creates incorrect (soft negative) sentence pairs
random.shuffle(soft_negatives)

# Create the training triplets
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):

    # Add the premise as the anchor sentence
    train_dataset["anchor"].append(row["premise"])

    # Add the correct hypothesis as the positive sentence
    train_dataset["positive"].append(row["hypothesis"])

    # Add the randomly shuffled hypothesis as the negative sentence
    train_dataset["negative"].append(soft_negative)

# Convert the dictionary into a Hugging Face Dataset
train_dataset = Dataset.from_dict(train_dataset)

In [ ]:
    # Create the STSB Evaluator

In [ ]:
# Import the evaluator used to evaluate the embedding model
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the validation split of the STSB (Semantic Textual Similarity Benchmark) dataset
val_sts = load_dataset("glue", "stsb", split="validation")

# Create an evaluator to measure the semantic similarity performance of the embedding model
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence in each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence in each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize the similarity scores from the range 0–5 to 0–1
    scores=[score / 5 for score in val_sts["label"]],

    # Use cosine similarity to compare the sentence embeddings
    main_similarity="cosine"
)

In [ ]:
    # Train the Embedding Model Using Multiple Negatives Ranking Loss

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
output_dir="mnrloss_embedding_model",
num_train_epochs=1,
per_device_train_batch_size=32,
per_device_eval_batch_size=32,
warmup_steps=100,
fp16=
True,
eval_steps=100,
logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
model=embedding_model,
args=args,
train_dataset=train_dataset,
loss=train_loss,
evaluator=evaluator
)

trainer.train()

In [ ]:
# Evaluate our trained model
evaluator(embedding_model)

                   Fine-Tuning an Embedding Model
                   Supervised

In [ ]:
!pip uninstall -y datasets huggingface_hub
!pip install "datasets==2.18.0" "huggingface_hub==0.23.5"

In [ ]:
    # Load the Training and Evaluation Datasets

In [ ]:
# Import the function to load datasets and the evaluator for embedding models
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the first 50,000 training examples from the MNLI dataset
# Labels: 0 = Entailment, 1 = Neutral, 2 = Contradiction
train_dataset = load_dataset(
    "glue", "mnli", split="train"
).select(range(50_000))

# Remove the unnecessary index column
train_dataset = train_dataset.remove_columns("idx")

# Load the validation split of the STSB (Semantic Textual Similarity Benchmark) dataset
val_sts = load_dataset("glue", "stsb", split="validation")

# Create an evaluator to measure the semantic similarity performance of the embedding model
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence in each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence in each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize the similarity scores from the range 0–5 to 0–1
    scores=[score / 5 for score in val_sts["label"]],

    # Use cosine similarity to compare the sentence embeddings
    main_similarity="cosine"
)

In [ ]:
# Model and loss function

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

In [ ]:
   # Configure the Fine-Tuning Settings

In [4]:
# Define the training arguments
args = SentenceTransformerTrainingArguments(
output_dir="finetuned_embedding_model",
num_train_epochs=1,
per_device_train_batch_size=32,
per_device_eval_batch_size=32,
warmup_steps=100,
fp16=
True
,
eval_steps=100,
logging_steps=100,
)

In [ ]:
    # Fine-Tune the Pretrained Embedding Model

In [ ]:
# Train model
trainer = SentenceTransformerTrainer(
model=embedding_model,
args=args,
train_dataset=train_dataset,
loss=train_loss,
evaluator=evaluator
)
trainer.train()

In [ ]:
# Evaluate our trained model
evaluator(embedding_model)

                           Augmented SBERT

In [ ]:
    # Prepare the Gold Dataset

In [ ]:
# Import the Pandas library for handling tabular data
import pandas as pd

# Import tqdm to display a progress bar
from tqdm import tqdm

# Import the Dataset class and the function to load datasets
from datasets import load_dataset, Dataset

# Import the InputExample class used to create training examples
from sentence_transformers import InputExample

# Import the dataloader that avoids duplicate sentence pairs in a batch
from sentence_transformers.datasets import NoDuplicatesDataLoader

# Load the first 10,000 training examples from the MNLI dataset
dataset = load_dataset(
    "glue", "mnli", split="train"
).select(range(10_000))

# Map the original labels to similarity labels

mapping = {2: 0, 1: 0, 0: 1}

# Create InputExample objects for training the Cross-Encoder
gold_examples = [
    InputExample(

        # Store the premise and hypothesis as a sentence pair
        texts=[row["premise"], row["hypothesis"]],

        # Assign the mapped label to the sentence pair
        label=mapping[row["label"]]
    )

    # Repeat for every row in the dataset and show the progress
    for row in tqdm(dataset)
]

# Create a dataloader with a batch size of 32
gold_dataloader = NoDuplicatesDataLoader(
    gold_examples,
    batch_size=32
)

# Create a Pandas DataFrame from the dataset
gold = pd.DataFrame({

    # Store the premise as the first sentence
    "sentence1": dataset["premise"],

    # Store the hypothesis as the second sentence
    "sentence2": dataset["hypothesis"],

    # Store the mapped labels
    "label": [mapping[label] for label in dataset["label"]]
})

In [ ]:
    # STEP 1
    # Train the Cross-Encoder

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
# Train a cross-encoder on the gold dataset
cross_encoder = CrossEncoder("bert-base-uncased", num_labels=2)
cross_encoder.fit(
train_dataloader=gold_dataloader,
epochs=1,
show_progress_bar=True,
warmup_steps=100,
use_amp=False
)



In [ ]:
      # STEP 2
      # Prepare the Silver Dataset

In [10]:
# Load the remaining 40,000 MNLI sentence pairs as the unlabeled silver dataset
silver = load_dataset(
    "glue", "mnli", split="train"
).select(range(10_000, 50_000))

# Create a list of (premise, hypothesis) sentence pairs for prediction
pairs = list(zip(silver["premise"], silver["hypothesis"]))

In [ ]:
    # STEP 3
    # Generate Labels for the Silver Dataset

In [ ]:
# Import the NumPy library for numerical operations
import numpy as np

# Predict labels for the silver sentence pairs using the fine-tuned Cross-Encoder
output = cross_encoder.predict(
    pairs,
    apply_softmax=True,
    show_progress_bar=True
)

# Create a Pandas DataFrame with the predicted labels
silver = pd.DataFrame({

    # Store the premise as the first sentence
    "sentence1": silver["premise"],

    # Store the hypothesis as the second sentence
    "sentence2": silver["hypothesis"],

    # Assign the predicted label with the highest probability
    "label": np.argmax(output, axis=1)
})

In [ ]:
   # Create the Augmented Training Dataset and STSB Evaluator

In [14]:
# Combine the gold and silver datasets into a single dataset
data = pd.concat([gold, silver], ignore_index=True, axis=0)

# Remove duplicate sentence pairs from the combined dataset
data = data.drop_duplicates(
    subset=["sentence1", "sentence2"],
    keep="first"
)

# Convert the Pandas DataFrame into a Hugging Face Dataset
train_dataset = Dataset.from_pandas(
    data,
    preserve_index=False
)

# Import the evaluator used to evaluate the embedding model
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the validation split of the STSB dataset
val_sts = load_dataset("glue", "stsb", split="validation")

# Create an evaluator for measuring semantic similarity
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence in each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence in each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize the similarity scores from 0–5 to 0–1
    scores=[score / 5 for score in val_sts["label"]],

    # Compare embeddings using cosine similarity
    main_similarity="cosine"
)

In [ ]:
    # Train the Augmented SBERT Model

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer("bert-base-uncased")

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
output_dir="augmented_embedding_model",
num_train_epochs=1,
per_device_train_batch_size=32,
per_device_eval_batch_size=32,
warmup_steps=100,
fp16=True,
eval_steps=100,
logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
model=embedding_model,
args=args,
train_dataset=train_dataset,
loss=train_loss,
evaluator=evaluator
)
trainer.train()

In [ ]:
evaluator(embedding_model)

                           Unsupervised Learning

In [ ]:
            # Transformer-Based Sequential Denoising Auto-Encoder

In [ ]:
   # Download the Required NLTK Tokenizers

In [ ]:
# Import the NLTK library for natural language processing tasks
import nltk

# Download the 'punkt' tokenizer used to split text into sentences and words
nltk.download("punkt")

# If you're using a newer version of NLTK may also need punk_tab:
# Download the additional Punkt tokenizer data required by newer NLTK versions
nltk.download("punkt_tab")


In [ ]:
    # Prepare the TSDAE Training Dataset

In [ ]:
# Import tqdm to display a progress bar during dataset creation
from tqdm import tqdm

# Import the Dataset class and the function to load datasets
from datasets import Dataset, load_dataset

# Import the dataset class used to create damaged and original sentence pairs for TSDAE training
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

# Load the first 25,000 training examples from the MNLI dataset
mnli = load_dataset("glue", "mnli", split="train").select(range(25_000))

# Combine all premise and hypothesis sentences into a single list of sentences
flat_sentences = mnli["premise"] + mnli["hypothesis"]

# Remove duplicate sentences and automatically create damaged-original sentence pairs
damaged_data = DenoisingAutoEncoderDataset(list(set(flat_sentences)))

# Create a dictionary to store the damaged sentences and their corresponding original sentences
train_dataset = {
    "damaged_sentence": [],
    "original_sentence": []
}

# Iterate through each damaged-original sentence pair and store them in the dictionary
for data in tqdm(damaged_data):

    # Store the damaged (noisy) version of the sentence
    train_dataset["damaged_sentence"].append(data.texts[0])

    # Store the corresponding original sentence
    train_dataset["original_sentence"].append(data.texts[1])

# Convert the dictionary into a Hugging Face Dataset for model training
train_dataset = Dataset.from_dict(train_dataset)

In [ ]:
train_dataset[0]

In [ ]:
   # Create the STSB Evaluator

In [29]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
sentences1=val_sts["sentence1"],
sentences2=val_sts["sentence2"],
scores=[score/5 for score in val_sts["label"]],
main_similarity="cosine"
)

In [ ]:
   # Build the TSDAE Embedding Model

In [31]:
# Import the building blocks used to create a custom SentenceTransformer model
from sentence_transformers import models, SentenceTransformer

# Load the BERT transformer model to generate token-level embeddings
word_embedding_model = models.Transformer("bert-base-uncased")

# Create a pooling layer that uses the [CLS] token as the sentence embedding
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    "cls"
)

# Combine the transformer and pooling layer into a complete SentenceTransformer model
embedding_model = SentenceTransformer(
    modules=[word_embedding_model, pooling_model]
)

In [ ]:
    # Configure the TSDAE Loss Function

In [ ]:
# Import the loss functions used for training SentenceTransformer models
from sentence_transformers import losses

# Create the Denoising Auto-Encoder (TSDAE) loss for training the embedding model
train_loss = losses.DenoisingAutoEncoderLoss(

    # The embedding model used as the encoder
    embedding_model,

    # Share the encoder and decoder weights during training
    tie_encoder_decoder=True
)

# Move the decoder model to the GPU for faster training
train_loss.decoder = train_loss.decoder.to("cuda")

In [ ]:
    # Train the TSDAE Embedding Model

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
# Define the training arguments
args = SentenceTransformerTrainingArguments(
output_dir="tsdae_embedding_model",
num_train_epochs=1,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
warmup_steps=100,
fp16=True,
eval_steps=100,
logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
model=embedding_model,
args=args,
train_dataset=train_dataset,
loss=train_loss,
evaluator=evaluator
)
trainer.train()

In [ ]:
# Evaluate our trained model
evaluator(embedding_model)